In [1]:
import numpy as np
import pandas as pd
import pycountry
from scipy import stats
import statsmodels.api as sm
import pyfixest as pf

# Figure 1, Figure S2

In [2]:
usecols = [
    "accepted_date", "ai_use_intro_discussion",
    "first_author_country", "last_author_country",
    "first_author_english_country", "last_author_english_country",
]
df = pd.read_csv( "../../data/processed/paper_level_analysis.csv", usecols=usecols)
df["accepted_date"] = pd.to_datetime(
    df["accepted_date"].astype("string").str.strip(),
    format="%Y-%m-%d", errors="coerce",
)
df["accepted_year"] = df["accepted_date"].dt.year
df = df[df["accepted_year"].between(2021, 2024)].copy()

country_epi = pd.read_csv( "../../data/reference/country_epi.tsv", sep="\t").rename(
    columns={"score": "epi_score"}
)
country_language = pd.read_csv("../../data/reference/country_language_type.tsv", sep="\t")

annotated_countries = {
    "SG", "CN", "NL", "IN", "FR",  "JP", "CM",
    "RO", "KR", "PT", "ES", "NO", "TH", "DZ", "MY",
}
panel_b_columns = [
    "record_type", "country", "country_name", "language_family",
    "epi_score", "ai_content_change", "paper_count", "pre_period_n",
    "post_period_n", "annotate_country", "regression_x",
    "predicted_epi", "ci95_lower", "ci95_upper",
]

In [3]:
def calculate_author_figure(author_order):
    if author_order not in {"first", "last"}:
        raise ValueError("author_order must be 'first' or 'last'")

    figure_name = "figure1" if author_order == "first" else "figure_s2"
    country_col = f"{author_order}_author_country"
    english_col = f"{author_order}_author_english_country"

    # Panel a: monthly mean and 95% confidence interval
    panel_a = df.dropna(
        subset=["accepted_date", "ai_use_intro_discussion", english_col]
    ).copy()
    panel_a["accepted_month"] = (
        panel_a["accepted_date"].dt.to_period("M").dt.to_timestamp()
    )
    panel_a = (
        panel_a.groupby(["accepted_month", english_col])["ai_use_intro_discussion"]
        .agg(n="count", mean_ai_content="mean", std_ai_content="std")
        .reset_index()
        .rename(columns={english_col: "english_country"})
    )
    panel_a["english_country"] = panel_a["english_country"].astype(float)
    panel_a["se_ai_content"] = panel_a["std_ai_content"] / np.sqrt(panel_a["n"])
    panel_a["t_critical"] = panel_a["n"].map(
        lambda n: stats.t.ppf(0.975, n - 1) if n >= 2 else np.nan
    )
    panel_a["ci95_lower"] = (
        panel_a["mean_ai_content"]
        - panel_a["t_critical"] * panel_a["se_ai_content"]
    )
    panel_a["ci95_upper"] = (
        panel_a["mean_ai_content"]
        + panel_a["t_critical"] * panel_a["se_ai_content"]
    )
    panel_a["group_label"] = panel_a["english_country"].map(
        {1.0: "English-speaking country", 0.0: "Non-English-speaking country"}
    )
    panel_a = panel_a.drop(columns="t_critical").sort_values(
        ["accepted_month", "english_country"]
    )

    # Panel b/c: country change, EPI regression, and map data
    country_data = df.dropna(
        subset=[country_col, "accepted_year", "ai_use_intro_discussion"]
    ).copy()
    country_data[country_col] = (
        country_data[country_col].astype("string").str.strip().str.upper()
    )
    country_data = country_data[
        ~country_data[country_col].str.contains(",", regex=False, na=False)
    ]
    country_data = country_data[
        country_data[country_col].str.fullmatch(r"[A-Z]{2}", na=False)
    ]

    yearly_counts = (
        country_data.groupby([country_col, "accepted_year"])
        .size().unstack(fill_value=0)
        .reindex(columns=[2021, 2022, 2023, 2024], fill_value=0)
    )
    valid_countries = yearly_counts.index[(yearly_counts >= 100).all(axis=1)]
    country_data = country_data[country_data[country_col].isin(valid_countries)]

    country_rows = []
    for country, country_df in country_data.groupby(country_col):
        pre = country_df[country_df["accepted_year"].isin([2021, 2022])][
            "ai_use_intro_discussion"
        ].dropna()
        post = country_df[country_df["accepted_year"].isin([2023, 2024])][
            "ai_use_intro_discussion"
        ].dropna()
        country_rows.append(
            {
                "country": country,
                "ai_content_change": post.mean() - pre.mean(),
                "paper_count": len(country_df),
                "pre_period_n": len(pre),
                "post_period_n": len(post),
            }
        )

    country_changes = pd.DataFrame(country_rows).sort_values("country")
    country_points = (
        country_changes.merge(country_epi[["country", "epi_score"]], on="country")
        .merge(country_language[["country", "language_family"]], on="country")
        .dropna(subset=["ai_content_change", "epi_score", "language_family"])
        .copy()
    )
    country_points["record_type"] = "country"
    country_points["country_name"] = country_points["country"].map(
        lambda x: pycountry.countries.get(alpha_2=x).name
        if pycountry.countries.get(alpha_2=x) is not None else x
    )
    country_points["annotate_country"] = country_points["country"].isin(
        annotated_countries
    )

    x = country_points["ai_content_change"].astype(float).to_numpy()
    y = country_points["epi_score"].astype(float).to_numpy()
    model = sm.OLS(y, sm.add_constant(x)).fit()
    regression_x = np.linspace(x.min(), x.max(), 200)
    prediction = model.get_prediction(
        sm.add_constant(regression_x)
    ).summary_frame(alpha=0.05)
    regression_rows = pd.DataFrame(
        {
            "record_type": "regression",
            "regression_x": regression_x,
            "predicted_epi": prediction["mean"].to_numpy(),
            "ci95_lower": prediction["mean_ci_lower"].to_numpy(),
            "ci95_upper": prediction["mean_ci_upper"].to_numpy(),
        }
    )
    panel_b = pd.concat(
        [
            country_points.reindex(columns=panel_b_columns),
            regression_rows.reindex(columns=panel_b_columns),
        ],
        ignore_index=True,
    )

    panel_c = country_changes.copy()
    panel_c["country_iso3"] = panel_c["country"].map(
        lambda x: pycountry.countries.get(alpha_2=x).alpha_3
        if pycountry.countries.get(alpha_2=x) is not None else None
    )
    panel_c = panel_c.dropna(subset=["country_iso3"])[
        [
            "country", "country_iso3", "ai_content_change",
            "paper_count", "pre_period_n", "post_period_n",
        ]
    ].sort_values("country")

    panel_a.to_csv(
         f"../../results/figure_data/data_panel_a_{figure_name}.csv",
        index=False, date_format="%Y-%m-%d",
    )
    panel_b.to_csv( f"../../results/figure_data/data_panel_b_{figure_name}.csv", index=False)
    panel_c.to_csv(f"../../results/figure_data/data_panel_c_{figure_name}.csv", index=False)
    return panel_a, panel_b, panel_c

In [4]:
figure1_a, figure1_b, figure1_c = calculate_author_figure("first")

In [5]:
figure_s2_a, figure_s2_b, figure_s2_c = calculate_author_figure("last")

# Figure 3

In [6]:
first_author_level_data = pd.read_csv('../../data/processed/first_author_analysis.csv')
last_author_level_data = pd.read_csv('../../data/processed/last_author_analysis.csv')

ai_values = [1, 0]
quantile_labels = [f'{i * 20}-{(i + 1) * 20}%' for i in range(5)]

def mean_ci(series, conf=0.95):
    arr = series.dropna().values
    n = len(arr)
    mean = np.mean(arr)
    if n > 1:
        se = stats.sem(arr)
        h = se * stats.t.ppf((1 + conf) / 2., n-1)
    else:
        h = np.nan
    return mean, h

def get_means_cis(data, value_col, group_col, group_values, sub_group_col, sub_group_values, group_label="author_group"):
    recs = []
    for eng in group_values:
        for sub in sub_group_values:
            subset = data[
                (data[group_col] == eng) &
                (data[sub_group_col] == sub)
            ][value_col]
            m, c = mean_ci(subset)
            recs.append({
                "country": "English-speaking" if eng == 1 else "Non-English-speaking",
                "ai_author": "AI Author" if sub == 1 else "Non-AI Author",
                "mean_ai_use_change": m,
                "ci": c
            })
    return pd.DataFrame(recs)

# Save panel a/b summary data for Figure 3.
means_first_df = get_means_cis(
    first_author_level_data, 'ai_use_change', 'author_english_country', [1, 0], 'ai_author', ai_values)
means_first_df['role'] = 'First Author'

means_last_df = get_means_cis(
    last_author_level_data, 'ai_use_change', 'author_english_country', [1, 0], 'ai_author', ai_values)
means_last_df['role'] = 'Corresponding Author'

data_figure3ab = pd.concat([means_first_df, means_last_df], ignore_index=True)
data_figure3ab.to_csv("../../results/figure_data/data_figure3ab.csv", index=False, encoding="utf-8-sig")

# Save panel c/d quantile summary data for Figure 4.
def get_quantile_results(df, role):
    df_ = df.copy()
    df_['ai_use_change_quantile'] = pd.qcut(df_['ai_use_change'], 5, labels=False, duplicates='drop')
    out = []
    for group_val in [1, 0]:
        country_str = "English-speaking" if group_val == 1 else "Non-English-speaking"
        df_group = df_[df_['author_english_country'] == group_val]
        grouped = df_group.groupby('ai_use_change_quantile')['pub_change']
        means = grouped.mean()
        counts = grouped.count()
        stds = grouped.std()
        conf_ints = 1.96 * stds / np.sqrt(counts)
        for q in means.index:
            out.append({
                'role': role,
                'country': country_str,
                'quantile': int(q),
                'quantile_label': quantile_labels[q],
                'mean_productivity_change': means[q],
                'ci': conf_ints[q],
                'n': counts[q]
            })
    return out

results_first = get_quantile_results(first_author_level_data, 'First Author')
results_last = get_quantile_results(last_author_level_data, 'Corresponding Author')
data_figure4cd = pd.DataFrame(results_first + results_last)
data_figure4cd.to_csv("../../results/figure_data/data_figure4cd.csv", index=False, encoding="utf-8-sig")


def get_zero_quantile_and_mean(df):
    df_temp = df.copy()
    df_temp['ai_use_change_quantile'] = pd.qcut(df_temp['ai_use_change'], 5, labels=False, duplicates='drop')
    zero_mask = df_temp['ai_use_change'] == 0
    if zero_mask.sum() == 0:
        idx = (df_temp['ai_use_change'] - 0).abs().idxmin()
        zero_quantile = int(df_temp.loc[idx, 'ai_use_change_quantile'])
        mean_paper_increase = df_temp.loc[idx, 'pub_change']
    else:
        zero_quantile = int(df_temp.loc[zero_mask, 'ai_use_change_quantile'].mode().iloc[0])
        mean_paper_increase = df_temp.loc[zero_mask, 'pub_change'].mean()
    return zero_quantile, mean_paper_increase

zero_quantile_first, mean_zero_first = get_zero_quantile_and_mean(first_author_level_data)
zero_quantile_last, mean_zero_last = get_zero_quantile_and_mean(last_author_level_data)

with open("../../results/figure_data/figure4cd_zeroline_info.txt", "w") as f:
    f.write(f"first_author_zero_quantile: {zero_quantile_first}, mean: {mean_zero_first}\n")
    f.write(f"corresponding_author_zero_quantile: {zero_quantile_last}, mean: {mean_zero_last}\n")
